<div dir="rtl">

# آموزش آماده‌سازی داده‌های NHANES

این دفترچه مرحلهٔ اول پژوهش را قدم‌به‌قدم اجرا می‌کند: دریافت داده، اتصال فایل‌ها، انتخاب افراد، خواندن کدها و گزارش گمشدگی. هر ردیف یک نفر و هر ستون یک ویژگی است. در این مرحله مدل یا SHAP محاسبه نمی‌شود.

**روش اجرا:** در Google Colab از File > Upload notebook همین فایل را باز کن و سلول‌ها را از بالا به پایین اجرا کن. اگر فایل‌های داده در پوشهٔ `data/raw` حاضر نباشند، از CDC دانلود می‌شوند؛ پس برای اجرای مستقل اینترنت لازم است. اگر کل بسته را روی رایانه باز کرده‌ای، فایل‌های اصلی داخل آن وجود دارند و دوباره دانلود نمی‌شوند.

**نسخهٔ اصلی این اجرا:** ۱۱ پیش‌بین از پیش‌نویس مقاله، افراد ۲۰ ساله و بالاتر، شرکت در معاینه و پاسخ بله یا خیر به سؤال دیابت. فشار خون با میانگین خواندن‌های موجود ساخته می‌شود. این انتخاب عملیاتی باید با استاد نهایی شود. فایل `prepare_nhanes.py` گزارش‌های ممیزی بیشتری از همین مراحل تولید می‌کند.

خروجی‌های زیر از اجرای واقعی همین سلول‌ها روی فایل‌های CDC ایجاد شده‌اند. برای تغییر تعریف پژوهش، ابتدا خروجی فعلی را نگه دار و سپس یک نسخهٔ جدا اجرا کن.

</div>


<div dir="rtl">

## ۱ آماده‌کردن ابزارها

`pandas` ابزار کار با جدول است. `NaN` یعنی مقدار ثبت نشده است؛ صفر با مقدار ثبت‌نشده فرق دارد. برای اجرای محلی Python 3.11 یا بالاتر و بسته‌های فایل `requirements.txt` لازم است. اگر در محیط شخصی کتابخانه‌ای نصب نیست، در ترمینال `python -m pip install -r requirements.txt` را اجرا کن. Colab معمولاً این کتابخانه‌ها را دارد.

</div>


In [1]:
from pathlib import Path
import urllib.request
import json
import pandas as pd
import numpy as np

cwd = Path.cwd()
ROOT = cwd if (cwd / 'data/raw').exists() else cwd / 'nhanes_stage1'
RAW = ROOT / 'data/raw'
OUT = ROOT / 'notebook_outputs'
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('pandas:', pd.__version__, '| numpy:', np.__version__)
print('مسیر خروجی:', OUT.name)


pandas: 2.2.3 | numpy: 2.3.5
مسیر خروجی: notebook_outputs


<div dir="rtl">

## ۲ دریافت شش فایل

`DEMO_J` مشخصات افراد، `DIQ_J` سؤال‌های دیابت، `BMX_J` اندازه‌های بدن، `BPX_J` فشار خون، `SMQ_J` سیگار و `PAQ_J` فعالیت بدنی را دارد. پسوند `xpt` قالب فایل داده است. برای معنی هر ستون باید صفحهٔ راهنمای همان فایل را بخوانیم.

ستون `SEQN` شناسهٔ فرد است. اگر شناسهٔ یک فرد در دو فایل یکسان باشد، اطلاعات آن دو ردیف متعلق به همان فرد است. هیچ فایل آزمایشگاهی در این اجرای اصلی استفاده نمی‌شود.

یک نکتهٔ فنی از کنترل واقعی داده: بعضی نسخه‌های pandas صفرهای XPT را به صورت عدد بسیار کوچک `2**-260` می‌خوانند. این خطای شناخته‌شده در فایل‌های این بسته با بایت‌های خام کنترل شد. خط اصلاح فقط همین مقدار دقیق را به صفر برمی‌گرداند؛ صفر را گمشده نمی‌کند و بقیه اعداد کوچک را تغییر نمی‌دهد. [گزارش توسعه‌دهندگان pandas](https://github.com/pandas-dev/pandas/issues/30051)

</div>


In [2]:
BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles'
files = ['DEMO_J', 'DIQ_J', 'BMX_J', 'BPX_J', 'SMQ_J', 'PAQ_J']
tables = {}
for name in files:
    path = RAW / f'{name}.xpt'
    if not path.exists():
        with urllib.request.urlopen(f'{BASE}/{name}.xpt', timeout=60) as response:
            content = response.read()
        if not content.startswith(b'HEADER RECORD'):
            raise ValueError(f'{name}: پاسخ سرور فایل XPT نیست')
        path.write_bytes(content)
    tables[name] = pd.read_sas(path, format='xport')
    # Exact verified XPT zero-decoding artifact, not a missing-value code.
    tables[name] = tables[name].replace(2.0**-260, 0.0)
    assert tables[name]['SEQN'].notna().all()
    assert tables[name]['SEQN'].is_unique
    print(name, '| افراد:', len(tables[name]), '| ستون‌ها:', len(tables[name].columns))


DEMO_J | افراد: 9254 | ستون‌ها: 46
DIQ_J | افراد: 8897 | ستون‌ها: 54
BMX_J | افراد: 8704 | ستون‌ها: 21
BPX_J | افراد: 8704 | ستون‌ها: 21
SMQ_J | افراد: 6724 | ستون‌ها: 37
PAQ_J | افراد: 5856 | ستون‌ها: 17


<div dir="rtl">

## ۳ انتخاب ستون‌ها و اتصال فایل‌ها

با `merge` اطلاعات یک فرد را کنار هم می‌گذاریم. `how='left'` یعنی همهٔ افراد فایل جمعیت‌شناختی حفظ شوند. اگر بخشی از اطلاعات کسی وجود نداشته باشد، سلول مربوطه خالی می‌ماند. `validate='one_to_one'` مانع اتصال اشتباه شناسه‌های تکراری می‌شود.

</div>


In [3]:
columns = {
    'DEMO_J': ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'RIDRETH3',
               'DMDEDUC2', 'INDFMPIR', 'RIDSTATR', 'RIDEXPRG',
               'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA'],
    'DIQ_J': ['SEQN', 'DIQ010'],
    'BMX_J': ['SEQN', 'BMXBMI', 'BMXWAIST'],
    'BPX_J': ['SEQN'] + [f'{p}{i}' for p in ['BPXSY', 'BPXDI'] for i in range(1, 5)],
    'SMQ_J': ['SEQN', 'SMQ020'],
    'PAQ_J': ['SEQN', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665'],
}
data = tables['DEMO_J'][columns['DEMO_J']].copy()
for name in files[1:]:
    assert tables[name]['SEQN'].isin(data['SEQN']).all()
    data = data.merge(tables[name][columns[name]], on='SEQN', how='left', validate='one_to_one')
data['SEQN'] = data['SEQN'].astype('int64')
assert len(data) == len(tables['DEMO_J'])
print('تعداد افراد پس از اتصال:', len(data))


تعداد افراد پس از اتصال: 9254


<div dir="rtl">

## ۴ معنی هدف پژوهش

هدف، پاسخ `DIQ010` است: آیا پزشک یا متخصص سلامت تا به حال به فرد گفته دیابت دارد؟ در تحلیل اصلی، ۱ یعنی بله و ۲ یعنی خیر؛ ۳ پاسخ مرزی، ۷ امتناع و ۹ نمی‌دانم است. پاسخ مرزی و نامعلوم را هدف صفر حساب نمی‌کنیم. «خیر» ممکن است شامل دیابت تشخیص‌داده‌نشده هم باشد.

`RIDAGEYR` سن فرد است. `DIQ175C` در لینک فرستاده‌شده یعنی فرد سن را دلیل احساس خطر دیابت دانسته و سن واقعی نیست. این ستون وارد مدل ما نمی‌شود.

</div>


In [4]:
adult_mec = data['RIDAGEYR'].ge(20) & data['RIDSTATR'].eq(2)
print('پاسخ‌های خام DIQ010 در افراد ۲۰ ساله و بالاتر که معاینه شده‌اند:')
print(data.loc[adult_mec, 'DIQ010'].value_counts(dropna=False).sort_index().to_string())


پاسخ‌های خام DIQ010 در افراد ۲۰ ساله و بالاتر که معاینه شده‌اند:
DIQ010
1.0     836
2.0    4262
3.0     163
9.0       4


<div dir="rtl">

## ۵ تبدیل کدهای ویژه به مقدار گمشده

در بعضی سؤال‌ها، ۷ و ۹ پاسخ واقعی نیستند و باید `NaN` شوند. این قانون برای همهٔ ستون‌ها درست نیست؛ مثلاً کد ۷ در `RIDRETH3` یک گروه معتبر و عدد ۷ در سن یک سن واقعی است. بنابراین فقط ستون‌هایی را تغییر می‌دهیم که کدبوکشان این دو مقدار را عدم پاسخ معرفی کرده است.

`SMQ020` سابقهٔ مصرف دست‌کم ۱۰۰ نخ سیگار در طول عمر را می‌پرسد، نه مصرف فعلی. `PAQ605` فعالیت شدید در کار و کارهای خانه و حیاط را می‌پرسد، نه کل ورزش فرد.

</div>


In [5]:
special_code_columns = ['DMDEDUC2', 'SMQ020', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665']
for col in special_code_columns:
    data[col] = data[col].replace({7: np.nan, 9: np.nan})
print('فقط کدهای ویژهٔ ستون‌های مشخص‌شده به NaN تبدیل شدند.')


فقط کدهای ویژهٔ ستون‌های مشخص‌شده به NaN تبدیل شدند.


<div dir="rtl">

## ۶ خلاصه‌کردن خواندن‌های فشار خون

برای یک نفر ممکن است چند بار فشار خون اندازه‌گیری شده باشد. این پروژه میانگین خواندن‌های موجود ۱ تا ۴ را می‌گیرد. اگر هر چهار خالی باشند، میانگین هم خالی است. خالی‌بودن خواندن چهارم به‌تنهایی به معنای گمشده‌بودن ویژگی فشار خون نیست؛ خواندن چهارم معمولاً یک تلاش اضافی است.

این یک انتخاب مشخص برای پروژه است و نامش را «الگوریتم رسمی میانگین‌گیری NHANES» نمی‌گذاریم. صفر دیاستولیک در کدبوک مقدار مجاز است و خودکار حذف نمی‌شود. شمار خواندن‌ها را نیز ذخیره می‌کنیم.

</div>


In [6]:
for prefix, name in [('BPXSY', 'SBP'), ('BPXDI', 'DBP')]:
    bp = data[[f'{prefix}{i}' for i in range(1, 5)]]
    data[f'{name}_MEAN'] = bp.mean(axis=1)
    data[f'{name}_N_READINGS'] = bp.notna().sum(axis=1)
print('ستون‌های SBP_MEAN و DBP_MEAN ساخته شدند؛ هیچ جایگزینی انجام نشد.')


ستون‌های SBP_MEAN و DBP_MEAN ساخته شدند؛ هیچ جایگزینی انجام نشد.


<div dir="rtl">

## ۷ انتخاب گروه پژوهش

سه شرط داریم: سن حداقل ۲۰ سال، حضور در معاینه و پاسخ بله یا خیر به دیابت. شرط معاینه در این اجرا به علت استفاده از BMI، دور کمر و فشار خون اضافه شده و باید در روش مقاله نوشته شود. افراد با پیش‌بین گمشده را نگه می‌داریم. در تعریف اصلی شرط حذف بارداری اعمال نشده است.

بعد از انتخاب، هدف جدید `diabetes` برابر ۱ برای بله و صفر برای خیر می‌شود. شناسه و ستون خام `DIQ010` پیش‌بین نیستند.

</div>


In [7]:
age_ok = data['RIDAGEYR'].ge(20)
mec_ok = data['RIDSTATR'].eq(2)
target_ok = data['DIQ010'].isin([1, 2])
cohort = data.loc[age_ok & mec_ok & target_ok].copy()
cohort['diabetes'] = cohort['DIQ010'].map({1: 1, 2: 0}).astype('int64')
assert cohort['WTMEC2YR'].gt(0).all()
flow = pd.DataFrame({
    'مرحله': ['همه افراد', 'سن حداقل ۲۰', 'شرکت در معاینه', 'هدف بله یا خیر'],
    'تعداد': [len(data), int(age_ok.sum()), int((age_ok & mec_ok).sum()), len(cohort)]
})
print(flow.to_string(index=False))
print('\nتعداد پاسخ بله:', int(cohort['diabetes'].sum()))
print('تعداد پاسخ خیر:', int(cohort['diabetes'].eq(0).sum()))


         مرحله  تعداد
     همه افراد   9254
   سن حداقل ۲۰   5569
شرکت در معاینه   5265
هدف بله یا خیر   5098

تعداد پاسخ بله: 836
تعداد پاسخ خیر: 4262


<div dir="rtl">

## ۸ اندازه‌گیری گمشدگی

۱۱ ستون زیر پیش‌بین‌های اصلی‌اند. در محاسبهٔ گمشدگی، فقط این ستون‌ها را می‌شماریم؛ شناسه، هدف، وزن‌ها و ستون‌های کمکی در مخرج نمی‌آیند.

`isna()` برای هر سلول می‌گوید خالی است یا خیر. `any(axis=1)` می‌گوید آیا در این ردیف دست‌کم یک سلول خالی وجود دارد. `sum()` تعداد افراد دارای این وضعیت را حساب می‌کند. `mean()` سهم آن‌ها از همهٔ افراد را به دست می‌دهد.

</div>


In [8]:
features = ['RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR',
            'BMXBMI', 'BMXWAIST', 'SBP_MEAN', 'DBP_MEAN', 'SMQ020', 'PAQ605']
missing = cohort[features].isna()
missingness = pd.DataFrame({
    'variable': features,
    'missing_n': missing.sum().values,
    'missing_pct': (100 * missing.mean()).values,
})
any_missing_n = int(missing.any(axis=1).sum())
complete_n = int((~missing.any(axis=1)).sum())
missing_cells = int(missing.sum().sum())
print(missingness.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
print(f'\nدارای حداقل یک پیش‌بین گمشده: {any_missing_n} / {len(cohort)} = {100*any_missing_n/len(cohort):.2f}%')
print('افراد کامل:', complete_n)
print(f'سلول‌های گمشده: {missing_cells} / {missing.size} = {100*missing_cells/missing.size:.2f}%')
assert any_missing_n + complete_n == len(cohort)


variable  missing_n  missing_pct
RIDAGEYR          0         0.00
RIAGENDR          0         0.00
RIDRETH1          0         0.00
DMDEDUC2         11         0.22
INDFMPIR        672        13.18
  BMXBMI         88         1.73
BMXWAIST        319         6.26
SBP_MEAN        258         5.06
DBP_MEAN        258         5.06
  SMQ020          0         0.00
  PAQ605          6         0.12

دارای حداقل یک پیش‌بین گمشده: 1067 / 5098 = 20.93%
افراد کامل: 4031
سلول‌های گمشده: 1612 / 56078 = 2.87%


<div dir="rtl">

**تفسیر:** درصد افراد دارای گمشدگی با درصد سلول‌های گمشده فرق دارد. یک نفر با سه مقدار خالی، در شمارش افراد فقط یک بار حساب می‌شود. درصدهای این دفترچه توصیف غیروزن‌دار همین نمونه‌اند و برآورد شیوع در کل آمریکا نیستند.

## ۹ بررسی اینکه تعریف متغیرها چه اثری دارد

دو فایل پژوهشی دقیقاً یک فهرست ندارند. پروپوزال ۱۳ پیش‌بین شامل چهار سؤال فعالیت و `RIDRETH3` دارد و تحصیلات را در فهرست اصلی نمی‌آورد. پیش‌نویس مقاله ۱۱ پیش‌بین شامل تحصیلات، `RIDRETH1` و فقط `PAQ605` دارد. همچنین استفاده از اولین خواندن فشار خون با میانگین خواندن‌ها فرق می‌کند. این مقایسه نشان می‌دهد چرا کنار هر درصد باید تعریف دقیق نوشته شود.

</div>


In [9]:
proposal_features = ['RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'INDFMPIR', 'BMXBMI', 'BMXWAIST',
                     'SBP_MEAN', 'DBP_MEAN', 'SMQ020', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665']
first_bp_features = [c.replace('SBP_MEAN', 'BPXSY1').replace('DBP_MEAN', 'BPXDI1') for c in features]
comparisons = []
for name, cols in [('article11_mean_BP', features), ('proposal13_mean_BP', proposal_features),
                   ('article11_first_BP', first_bp_features)]:
    m = cohort[cols].isna().any(axis=1)
    comparisons.append({'definition': name, 'n': len(cohort), 'features': len(cols),
                        'any_missing_n': int(m.sum()), 'any_missing_pct': float(100*m.mean())})
print(pd.DataFrame(comparisons).to_string(index=False, float_format=lambda x: f'{x:.2f}'))


        definition    n  features  any_missing_n  any_missing_pct
 article11_mean_BP 5098        11           1067            20.93
proposal13_mean_BP 5098        13           1065            20.89
article11_first_BP 5098        11           1335            26.19


<div dir="rtl">

## ۱۰ کنترل کیفیت و نگاه به توزیع‌ها

شناسه‌ها باید یکتا، هدف مشخص و کدهای دسته‌ای معتبر باشند. حداقل، میانه و حداکثر متغیرهای عددی را می‌بینیم. مشاهدهٔ یک مقدار بزرگ به‌تنهایی مجوز حذف آن نیست. سن ۸۰ یعنی ۸۰ سال و بیشتر و نسبت درآمد ۵ یعنی ۵ یا بیشتر.

در پژوهش بعدی، ویژگی دسته‌ای مثل قومیت را با همین کد عددی به‌عنوان یک مقدار پیوسته وارد مدل نمی‌کنیم. پیش‌پردازش مناسب پس از تقسیم آموزش و آزمون و فقط با اطلاعات آموزش برازش می‌شود.

</div>


In [10]:
assert cohort['SEQN'].is_unique
assert cohort['diabetes'].isin([0, 1]).all()
assert cohort[['WTMEC2YR', 'SDMVPSU', 'SDMVSTRA']].notna().all().all()
for col, allowed in {'RIAGENDR': [1, 2], 'RIDRETH1': [1, 2, 3, 4, 5],
                     'DMDEDUC2': [1, 2, 3, 4, 5], 'SMQ020': [1, 2], 'PAQ605': [1, 2]}.items():
    assert cohort[col].dropna().isin(allowed).all(), col
numeric_features = ['RIDAGEYR', 'INDFMPIR', 'BMXBMI', 'BMXWAIST', 'SBP_MEAN', 'DBP_MEAN']
print(cohort[numeric_features].describe().round(2).to_string())
print('\nکنترل‌های شناسه، هدف، کدها و ستون‌های طرح نمونه‌گیری تأیید شدند.')


       RIDAGEYR  INDFMPIR   BMXBMI  BMXWAIST  SBP_MEAN  DBP_MEAN
count   5098.00   4426.00  5010.00   4779.00   4840.00   4840.00
mean      51.19      2.54    29.80    100.63    126.96     72.52
std       17.72      1.60     7.36     16.99     20.09     12.97
min       20.00      0.00    14.20     57.90     72.67      0.00
25%       36.00      1.19    24.80     88.70    112.67     65.33
50%       52.00      2.12    28.60     99.30    124.00     72.67
75%       65.00      4.07    33.50    111.00    137.33     80.00
max       80.00      5.00    86.20    169.50    238.00    135.33

کنترل‌های شناسه، هدف، کدها و ستون‌های طرح نمونه‌گیری تأیید شدند.


<div dir="rtl">

## ۱۱ ذخیره خروجی‌ها

فایل اصلی تمام افراد واجد شرایط را با گمشدگی حفظ‌شده دارد. فایل موارد کامل یک خروجی کمکی برای مقایسهٔ بعدی CCA است و جای فایل اصلی را نمی‌گیرد. ستون‌های وزن و طرح نمونه‌گیری برای تحلیل پیمایش حفظ می‌شوند. فقط فهرست `features` ورودی پیش‌بین مدل آینده است؛ `DIQ010`، شناسه، هدف، وزن‌ها و پرچم گمشدگی نباید تصادفی به آن اضافه شوند.

خروجی CSV در این دفترچه در پوشهٔ `notebook_outputs` نوشته می‌شود تا خروجی گزارش اصلی دست‌نخورده بماند. CSV را می‌توان در Excel باز کرد.

</div>


In [11]:
cohort['missing_predictor_count'] = missing.sum(axis=1).astype('int64')
cohort['complete_case_main'] = (~missing.any(axis=1)).astype('int64')
design_columns = ['WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA']
keep = ['SEQN', 'DIQ010', 'diabetes'] + features + design_columns + ['missing_predictor_count', 'complete_case_main']
cohort[keep].to_csv(OUT / 'analytic_dataset.csv', index=False, encoding='utf-8-sig')
cohort.loc[cohort['complete_case_main'].eq(1), keep].to_csv(OUT / 'complete_cases.csv', index=False, encoding='utf-8-sig')
missingness.to_csv(OUT / 'missingness_by_variable.csv', index=False, encoding='utf-8-sig')
flow.to_csv(OUT / 'cohort_flow.csv', index=False, encoding='utf-8-sig')
print('ذخیره شد: analytic_dataset.csv، complete_cases.csv، missingness_by_variable.csv، cohort_flow.csv')


ذخیره شد: analytic_dataset.csv، complete_cases.csv، missingness_by_variable.csv، cohort_flow.csv


<div dir="rtl">

## ۱۲ قبل از ارائه به استاد

با زبان خودت به این پنج سؤال جواب بده:

۱. چرا فایل‌ها را با `SEQN` متصل کردیم؟

۲. چرا پاسخ مرزی دیابت را صفر نکردیم؟

۳. چرا یک عدد ۷ در یک ستون می‌تواند معتبر و در ستون دیگر گمشده باشد؟

۴. چرا ۲۰٫۹۳ درصد افراد دارای گمشدگی با ۲٫۸۷ درصد سلول‌های گمشده یکسان نیست؟

۵. چرا فعلاً داده‌ها را با میانه یا KNN پر نکردیم؟

پاسخ‌های کوتاه: شناسه هویت مشترک فرد در فایل‌هاست؛ مرزی معادل پاسخ خیر نیست؛ معنی عدد از کدبوک هر ستون می‌آید؛ مخرج و واحد شمارش متفاوت‌اند؛ مدیریت گمشدگی موضوع پژوهش است و باید داخل دادهٔ آموزشی مقایسه شود.

**مرحلهٔ بعد:** نهایی‌کردن همین تعریف‌ها با استاد، ساخت تقسیم‌های یکسان آموزش و آزمون، مقایسهٔ CCA و میانه/نما و KNN عددی، سپس مدل‌های LR و RF و توضیحات SHAP. این دفترچه هیچ نتیجه‌ای دربارهٔ برتری مدل یا روش جایگزینی ارائه نمی‌کند.

## منابع

صفحات رسمی CDC که معنی ستون‌ها و فایل‌های اصلی را مشخص می‌کنند:

- [DEMO_J](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.htm)
- [DIQ_J](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DIQ_J.htm)
- [BMX_J](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.htm)
- [BPX_J](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BPX_J.htm)
- [SMQ_J](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/SMQ_J.htm)
- [PAQ_J](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/PAQ_J.htm)
- [راهنمای انتخاب وزن](https://wwwn.cdc.gov/nchs/nhanes/tutorials/weighting.aspx)

فهرست اصلی با پیش‌نویس مقالهٔ پیوست کاربر همسو شده و فهرست جایگزین از پروپوزال پیوست آمده است. انتخاب میانگین همهٔ خواندن‌ها و شرط معاینه تصمیم‌های اجرایی شفاف این بسته‌اند.

</div>
